# 02 · Lab tables after the load
Run the loader first (yours from Day 5, or the reference):
```bash
python de-lab/reference/python/ingest_landing.py
```
`de-lab/lab.duckdb` is **your** sandbox: write anything here. The warehouse is attached read-only as `wh`.

In [1]:
import pathlib, duckdb
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "warehouse.duckdb").exists())
wh = duckdb.connect(str(ROOT / "warehouse.duckdb"), read_only=True)   # production: ALWAYS read-only
def q(sql, con=None):
    return (con or wh).sql(sql)
lab = duckdb.connect(str(ROOT / 'de-lab/lab.duckdb'), read_only=True)
q("SELECT table_name, estimated_size AS approx_rows FROM duckdb_tables()", lab)

table_nameVARCHAR,approx_rowsBIGINT
quarantine_orders,21
raw_orders,4904
stg_orders,4858


## What got quarantined, and why

In [2]:
q("SELECT reject_reason, _source_file, COUNT(*) AS rows FROM quarantine_orders GROUP BY ALL ORDER BY rows DESC", lab)

reject_reasonVARCHAR,_source_fileVARCHAR,rowsBIGINT
missing customer_id,orders_2026-10-17.csv,10
orphan restaurant_id,orders_2026-10-19.csv,8
negative amount,orders_2026-10-17.csv,3


## Reconciliation: raw → stg (every row accounted for)

In [3]:
q("""WITH c AS (SELECT (SELECT COUNT(*) FROM raw_orders) AS raw_rows,
                  (SELECT COUNT(*) FROM quarantine_orders) AS quarantined,
                  (SELECT COUNT(*) FROM stg_orders) AS stg_rows)
SELECT *, raw_rows - quarantined - stg_rows AS deduplicated FROM c""", lab)

raw_rowsBIGINT,quarantinedBIGINT,stg_rowsBIGINT,deduplicatedBIGINT
4904,21,4858,25


## Daily orders, including the gap on 20 Oct and the late rows

In [4]:
q("""SELECT order_date, COUNT(*) AS orders, COUNT(*) FILTER (WHERE is_late_arriving) AS late_rows
FROM stg_orders GROUP BY 1 ORDER BY 1""", lab)

order_dateDATE,ordersBIGINT,late_rowsBIGINT
2026-10-15,795,0
2026-10-16,790,0
2026-10-17,818,0
2026-10-18,823,0
2026-10-19,836,30
2026-10-21,796,0


### Exercise
Ask Claude to add a dbt test that would have caught the missing 20 Oct partition.